# LeetCode #17 — Letter Combinations of a Phone Number: nhập môn Backtracking

**Mức độ:** Medium · **Chủ đề:** Hash Table, String, Backtracking, Recursion

Skeleton trên LeetCode:

```python
class Solution:
    def letterCombinations(self, digits: str) -> List[str]:
```

Mục tiêu của notebook là giúp bạn hiểu **kỹ thuật quay lui (Backtracking)** — cách sinh ra **mọi tổ hợp** bằng cách "chọn → đi tiếp → bỏ chọn", và cách hình dung nó như một **cây quyết định**.

---

## Sau notebook này, bạn sẽ nắm được

1. Vì sao không thể giải bài này bằng một số **cố định** vòng `for` lồng nhau.
2. Cách xây kết quả **từng chữ số một** bằng vòng lặp (cách lặp).
3. Đệ quy (recursion) là gì: **trường hợp cơ sở** và **bước đệ quy**.
4. Mẫu Backtracking chuẩn: `path.append` → gọi đệ quy → `path.pop`.
5. Cách vẽ **cây quyết định** và đếm số kết quả.
6. Dùng `itertools.product` cho gọn.

# 1. Bài toán là gì?

Trên bàn phím điện thoại cũ, mỗi phím số `2`–`9` ứng với vài chữ cái:

| Phím | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|
| Chữ | abc | def | ghi | jkl | mno | pqrs | tuv | wxyz |

Cho chuỗi `digits` gồm các chữ số `2`–`9`. Trả về **mọi chuỗi chữ cái** có thể gõ ra từ dãy phím đó. Thứ tự kết quả tùy ý.

**Constraints (theo đề):** `digits.length <= 4`, mỗi `digits[i]` thuộc `['2', '9']`. Nếu `digits` rỗng, trả về `[]`.

---

## Ví dụ

```text
digits = "23"  ->  ["ad","ae","af","bd","be","bf","cd","ce","cf"]
digits = "2"   ->  ["a","b","c"]
digits = ""    ->  []
```

Với `"23"`: chữ đầu chọn trong `abc` (3 cách), chữ sau chọn trong `def` (3 cách) → `3 × 3 = 9` chuỗi.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đệ quy (recursion)
2. Quay lui (backtracking)
3. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [1]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [2]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [3]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [4]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [5]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [6]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [7]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [8]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [9]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [10]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [11]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [12]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 247.68 ms | trên set: 0.120 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [13]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [14]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.4 Áp dụng vào bài này

Mỗi chữ số ứng với vài chữ cái (bảng tra `dict`). Liệt kê mọi tổ hợp chính là **quay lui**: tầng thứ `i` của cây quyết định chọn một chữ cái cho chữ số thứ `i`; tới khi đủ độ dài thì ghi lại. Đây là ví dụ "tích Descartes" — số kết quả là tích số chữ cái của từng chữ số.

# 3. Nền tảng

## 3.1 Bảng tra phím → chữ bằng `dict`

In [15]:
PHONE = {
    "2": "abc", "3": "def", "4": "ghi", "5": "jkl",
    "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz",
}
print(PHONE["7"])
print([PHONE[d] for d in "23"])

pqrs
['abc', 'def']


## 3.2 Khi biết trước số chữ số: vòng `for` lồng nhau

Với **đúng 2** chữ số, hai vòng lồng nhau là đủ:

In [16]:
digits = "23"
result = []
for a in PHONE[digits[0]]:
    for b in PHONE[digits[1]]:
        result.append(a + b)
print(result)

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


**Vấn đề:** số vòng lặp phải bằng **độ dài `digits`** — mà độ dài thay đổi theo input (0 tới 4). Ta không thể viết sẵn số vòng `for` cố định.

Cần một cách "lồng vòng lặp với độ sâu tùy ý". Có hai hướng:

1. **Lặp:** xây dần danh sách kết quả, mỗi chữ số "nhân" danh sách lên.
2. **Đệ quy / Backtracking:** một hàm tự gọi chính nó, mỗi lần gọi xử lý một chữ số.

# 4. Cách 1: xây dần bằng vòng lặp

Bắt đầu với `[""]` (một chuỗi rỗng). Với mỗi chữ số, **nối mỗi chuỗi hiện có với mỗi chữ cái** của phím đó:

```text
bắt đầu:      [""]
sau '2':      ["a", "b", "c"]
sau '3':      ["ad","ae","af","bd","be","bf","cd","ce","cf"]
```

In [17]:
from typing import List

def letter_combinations_iterative(digits: str) -> List[str]:
    if not digits:
        return []
    result = [""]
    for d in digits:
        result = [prefix + ch for prefix in result for ch in PHONE[d]]
        print(f"sau {d!r}: {result}")
    return result

letter_combinations_iterative("23")

sau '2': ['a', 'b', 'c']
sau '3': ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']

Cách này ngắn gọn và đúng. Nhưng nó **giữ toàn bộ danh sách trung gian**, và không tổng quát cho các bài cần **cắt tỉa** (dừng sớm một nhánh khi biết chắc nó không dẫn tới đáp án) — như #22 Generate Parentheses hay #39 Combination Sum.

Backtracking giải quyết được cả hai.

# 5. Đệ quy là gì?

Một hàm **đệ quy** là hàm gọi lại chính nó với bài toán **nhỏ hơn**. Mỗi hàm đệ quy cần:

1. **Trường hợp cơ sở (base case):** khi nào dừng, không gọi tiếp.
2. **Bước đệ quy:** đưa bài toán về bài nhỏ hơn và gọi lại.

Ví dụ đơn giản: in đếm ngược.

In [18]:
def countdown(n: int):
    if n == 0:                 # base case
        print("Hết!")
        return
    print(n)
    countdown(n - 1)           # bài nhỏ hơn

countdown(3)

3
2
1
Hết!


Thiếu base case → hàm gọi mãi mãi → Python báo `RecursionError`.

# 6. Cây quyết định

Với `digits = "23"`, hãy hình dung việc chọn chữ như đi xuống một cây:

```text
                     ""
          /          |          \
        "a"         "b"         "c"        <- chọn chữ cho '2'
       / | \       / | \       / | \
     ad  ae af   bd  be bf   cd  ce cf     <- chọn chữ cho '3'
```

- Mỗi **tầng** ứng với một chữ số.
- Mỗi **nhánh** ứng với một chữ cái được chọn.
- Mỗi **lá** (tầng cuối) là một kết quả.

Backtracking = **duyệt cây này theo chiều sâu (DFS)**: đi xuống một nhánh tới tận lá, ghi lại, rồi **quay lui** lên để thử nhánh khác.

# 7. Cách 2: Backtracking

Ta giữ:

- `index`: đang chọn chữ cho chữ số thứ mấy.
- `path`: list các chữ cái đã chọn trên đường đi từ gốc.

```python
def backtrack(index):
    if index == len(digits):           # tới lá
        result.append("".join(path))
        return
    for ch in PHONE[digits[index]]:
        path.append(ch)                # CHỌN
        backtrack(index + 1)           # ĐI TIẾP
        path.pop()                     # BỎ CHỌN (quay lui)
```

**Vì sao phải `pop`?** `path` được dùng chung cho mọi nhánh. Sau khi thử xong nhánh `'a'`, phải xóa `'a'` đi thì mới đặt `'b'` vào đúng vị trí đó.

In [19]:
def letter_combinations_backtrack(digits: str) -> List[str]:
    if not digits:
        return []

    result = []
    path = []

    def backtrack(index: int):
        if index == len(digits):
            result.append("".join(path))
            return
        for ch in PHONE[digits[index]]:
            path.append(ch)
            backtrack(index + 1)
            path.pop()

    backtrack(0)
    return result

print(letter_combinations_backtrack("23"))
print(letter_combinations_backtrack("2"))
print(letter_combinations_backtrack(""))

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']
['a', 'b', 'c']
[]


**Hàm lồng (nested function):** `backtrack` được định nghĩa **bên trong** hàm chính, nên nó đọc và sửa được `result`, `path`, `digits` mà không cần truyền vào. (Ta chỉ gọi `.append` / `.pop`, không gán lại biến, nên không cần `nonlocal`.)

# 8. Mô phỏng từng bước

In ra mỗi lần chọn / bỏ chọn, thụt lề theo độ sâu để thấy rõ cây:

In [20]:
def trace_backtrack(digits: str) -> List[str]:
    result, path = [], []

    def backtrack(index: int):
        indent = "    " * index
        if index == len(digits):
            print(f"{indent}LÁ -> ghi {''.join(path)!r}")
            result.append("".join(path))
            return
        for ch in PHONE[digits[index]]:
            path.append(ch)
            print(f"{indent}chọn {ch!r} cho {digits[index]!r}, path = {path}")
            backtrack(index + 1)
            path.pop()
            print(f"{indent}bỏ {ch!r}, path = {path}")

    backtrack(0)
    return result

print(trace_backtrack("23"))

chọn 'a' cho '2', path = ['a']
    chọn 'd' cho '3', path = ['a', 'd']
        LÁ -> ghi 'ad'
    bỏ 'd', path = ['a']
    chọn 'e' cho '3', path = ['a', 'e']
        LÁ -> ghi 'ae'
    bỏ 'e', path = ['a']
    chọn 'f' cho '3', path = ['a', 'f']
        LÁ -> ghi 'af'
    bỏ 'f', path = ['a']
bỏ 'a', path = []
chọn 'b' cho '2', path = ['b']
    chọn 'd' cho '3', path = ['b', 'd']
        LÁ -> ghi 'bd'
    bỏ 'd', path = ['b']
    chọn 'e' cho '3', path = ['b', 'e']
        LÁ -> ghi 'be'
    bỏ 'e', path = ['b']
    chọn 'f' cho '3', path = ['b', 'f']
        LÁ -> ghi 'bf'
    bỏ 'f', path = ['b']
bỏ 'b', path = []
chọn 'c' cho '2', path = ['c']
    chọn 'd' cho '3', path = ['c', 'd']
        LÁ -> ghi 'cd'
    bỏ 'd', path = ['c']
    chọn 'e' cho '3', path = ['c', 'e']
        LÁ -> ghi 'ce'
    bỏ 'e', path = ['c']
    chọn 'f' cho '3', path = ['c', 'f']
        LÁ -> ghi 'cf'
    bỏ 'f', path = ['c']
bỏ 'c', path = []
['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


Để ý thứ tự: `path` lớn lên khi đi xuống và nhỏ lại khi quay lên — giống một **ngăn xếp (stack)**. Thực ra Python cũng dùng một ngăn xếp các lời gọi hàm (call stack) để nhớ mỗi `backtrack` đang ở vòng lặp nào.

# 9. Cách 3: `itertools.product`

Thư viện chuẩn có sẵn hàm sinh **tích Descartes** — chính là "vòng for lồng nhau độ sâu tùy ý":

In [21]:
from itertools import product

groups = [PHONE[d] for d in "23"]
print(groups)
print(["".join(t) for t in product(*groups)])

['abc', 'def']
['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


`product(*groups)` với `groups = ["abc", "def"]` giống `product("abc", "def")` — dấu `*` "trải" list ra thành nhiều tham số.

Trong phỏng vấn, nên biết cách này **nhưng** vẫn phải viết được backtracking, vì nhiều bài cần cắt tỉa mà `product` không làm được.

# 10. Phiên bản đúng format LeetCode

In [22]:
class Solution:
    def letterCombinations(self, digits: str) -> List[str]:
        if not digits:
            return []
        phone = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
                 "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
        result, path = [], []

        def backtrack(index: int):
            if index == len(digits):
                result.append("".join(path))
                return
            for ch in phone[digits[index]]:
                path.append(ch)
                backtrack(index + 1)
                path.pop()

        backtrack(0)
        return result

In [23]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<22} PASS  ({len(actual)} kết quả) {actual[:6]}{' ...' if len(actual) > 6 else ''}")


sol = Solution()
check("Ví dụ 1: '23'", sol.letterCombinations("23"), ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"])
check("Ví dụ 2: '2'", sol.letterCombinations("2"), ["a", "b", "c"])
check("Rỗng", sol.letterCombinations(""), [])
check("Phím 4 chữ: '79'", sol.letterCombinations("79"),
      [a + b for a in "pqrs" for b in "wxyz"])
check("Lặp phím: '22'", sol.letterCombinations("22"),
      ["aa", "ab", "ac", "ba", "bb", "bc", "ca", "cb", "cc"])

Ví dụ 1: '23'          PASS  (9 kết quả) ['ad', 'ae', 'af', 'bd', 'be', 'bf'] ...
Ví dụ 2: '2'           PASS  (3 kết quả) ['a', 'b', 'c']
Rỗng                   PASS  (0 kết quả) []
Phím 4 chữ: '79'       PASS  (16 kết quả) ['pw', 'px', 'py', 'pz', 'qw', 'qx'] ...
Lặp phím: '22'         PASS  (9 kết quả) ['aa', 'ab', 'ac', 'ba', 'bb', 'bc'] ...


## Đối chiếu ba cách trên mọi input độ dài 0..3

In [24]:
import io, contextlib

all_inputs = [""] + ["".join(p) for k in range(1, 4) for p in product("23456789", repeat=k)]
for digits in all_inputs:
    with contextlib.redirect_stdout(io.StringIO()):          # tắt print của bản lặp
        it = letter_combinations_iterative(digits)
    bt = sol.letterCombinations(digits)
    pr = ["".join(t) for t in product(*[PHONE[d] for d in digits])] if digits else []
    assert sorted(it) == sorted(bt) == sorted(pr), digits
print(f"Đã kiểm tra {len(all_inputs)} input: ba cách khớp nhau")

Đã kiểm tra 585 input: ba cách khớp nhau


# 11. Vì sao đúng? Độ phức tạp?

**Đúng:** cây quyết định có đúng một đường đi từ gốc tới lá cho **mỗi** tổ hợp (mỗi tầng chọn đúng một chữ cái của chữ số tương ứng). DFS đi qua **mọi** đường như vậy đúng **một lần** → không thiếu, không trùng.

**Số kết quả:** tích số chữ cái của từng phím. Tệ nhất mọi phím là `7` hoặc `9` (4 chữ) → `4^n`.

Gọi `n = len(digits)`:

```text
Time:  O(4^n · n)   — 4^n lá, mỗi lá tốn O(n) để "".join(path)
Space: O(n) cho path và độ sâu đệ quy (không tính kết quả)
       Kết quả tự nó chiếm O(4^n · n)
```

Với `n ≤ 4`: tối đa `4^4 = 256` chuỗi — rất nhỏ.

# 12. Những lỗi dễ mắc

## Lỗi 1: Quên `path.pop()`

`path` sẽ dài mãi, kết quả sai.

In [25]:
def wrong_no_pop(digits: str) -> List[str]:
    result, path = [], []

    def backtrack(index: int):
        if index == len(digits):
            result.append("".join(path))
            return
        for ch in PHONE[digits[index]]:
            path.append(ch)
            backtrack(index + 1)
            # thiếu path.pop()

    backtrack(0)
    return result

print("Sai :", wrong_no_pop("23")[:4], "...")
print("Đúng:", letter_combinations_backtrack("23")[:4], "...")

Sai : ['ad', 'ade', 'adef', 'adefbd'] ...
Đúng: ['ad', 'ae', 'af', 'bd'] ...


## Lỗi 2: Lưu chính `path` (list) vào kết quả thay vì một bản sao

`result.append(path)` lưu **tham chiếu** tới cùng một list. Khi `path` bị `pop` hết, mọi phần tử của `result` đều trở thành list rỗng.

In [26]:
def wrong_append_reference(digits: str):
    result, path = [], []

    def backtrack(index: int):
        if index == len(digits):
            result.append(path)          # SAI: tham chiếu, không phải bản sao
            return
        for ch in PHONE[digits[index]]:
            path.append(ch)
            backtrack(index + 1)
            path.pop()

    backtrack(0)
    return result

print("Sai :", wrong_append_reference("23"))

Sai : [[], [], [], [], [], [], [], [], []]


Sửa: `result.append("".join(path))` (tạo chuỗi mới) hoặc `result.append(path[:])` / `list(path)` (tạo bản sao list).

## Lỗi 3: Quên trường hợp `digits = ""`

Nếu không kiểm tra, backtracking gọi `backtrack(0)` với `len(digits) == 0` → ngay lập tức "tới lá" và ghi `""` → trả về `[""]` thay vì `[]`.

In [27]:
def wrong_empty(digits: str) -> List[str]:
    result, path = [], []
    def backtrack(index):
        if index == len(digits):
            result.append("".join(path))
            return
        for ch in PHONE[digits[index]]:
            path.append(ch); backtrack(index + 1); path.pop()
    backtrack(0)
    return result

print("Sai :", wrong_empty(""), "| Đúng:", letter_combinations_backtrack(""))

Sai : [''] | Đúng: []


## Lỗi 4: Quên phím 7 và 9 có **4** chữ cái

Gõ nhầm bảng `"7": "pqr"` sẽ thiếu kết quả. Hãy kiểm tra bảng bằng một test có phím 7 và 9.

# 13. Mẹo nhận diện pattern

Nghĩ tới **Backtracking** khi đề yêu cầu:

- "Trả về **tất cả** các tổ hợp / hoán vị / tập con / cách chia…"
- Số lựa chọn ở mỗi bước nhỏ, độ sâu nhỏ (input nhỏ: `n ≤ 10`, `≤ 20`…) — dấu hiệu kết quả có thể tăng theo hàm mũ.

## Template Backtracking

```python
result, path = [], []

def backtrack(state):
    if <đã đủ / tới lá>:
        result.append(<bản sao của path>)
        return
    for choice in <các lựa chọn ở state này>:
        if <choice không hợp lệ>:      # cắt tỉa (nếu có)
            continue
        path.append(choice)            # chọn
        backtrack(<state mới>)         # đi tiếp
        path.pop()                     # bỏ chọn

backtrack(<state ban đầu>)
```

Các bài cùng họ: **#22 Generate Parentheses**, **#39 / #40 Combination Sum**, **#46 / #47 Permutations**, **#78 Subsets**, **#79 Word Search**.

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

1. `"79"` cho ra bao nhiêu chuỗi? `"234"` thì sao?
2. Với `"92"`, liệt kê 5 kết quả đầu tiên theo thứ tự backtracking sinh ra.

## Bài 2 — Đoán output

```python
print(list(product("ab", "xy")))
print(["".join(t) for t in product("ab", repeat=2)])
print(len(list(product("abc", "def", "ghi"))))
```

## Bài 3 — Đếm số kết quả mà không sinh ra chúng

Viết hàm trả về **số lượng** chuỗi mà `letterCombinations(digits)` sẽ sinh ra (rỗng → 0).

```python
count_combinations("23")   -> 9
count_combinations("79")   -> 16
count_combinations("")     -> 0
```

In [28]:
def exercise_3_count_combinations(digits: str) -> int:
    # TODO: nhân số chữ cái của từng phím
    pass


# assert exercise_3_count_combinations("23") == 9
# assert exercise_3_count_combinations("79") == 16
# assert exercise_3_count_combinations("") == 0

## Bài 4 — Điền chỗ trống: Backtracking

In [29]:
def exercise_4_letter_combinations(digits: str) -> List[str]:
    if not digits:
        return []
    result, path = [], []

    def backtrack(index: int):
        # TODO 1: base case — khi index == len(digits), ghi "".join(path) vào result
        # TODO 2: với mỗi ch trong PHONE[digits[index]]: append, gọi đệ quy, pop
        pass

    backtrack(0)
    return result


# assert sorted(exercise_4_letter_combinations("23")) == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
# assert exercise_4_letter_combinations("") == []

## Bài 5 — Biến thể: mọi chuỗi nhị phân độ dài `n`

Sinh mọi chuỗi gồm `'0'` và `'1'` có độ dài `n`, theo thứ tự tăng dần, bằng backtracking.

```python
binary_strings(2) -> ["00", "01", "10", "11"]
binary_strings(0) -> [""]
```

In [30]:
def exercise_5_binary_strings(n: int) -> List[str]:
    # TODO
    pass


# assert exercise_5_binary_strings(2) == ["00", "01", "10", "11"]
# assert exercise_5_binary_strings(0) == [""]
# assert len(exercise_5_binary_strings(5)) == 32

## Bài 6 — Mini challenge: cắt tỉa

Sinh mọi chuỗi nhị phân độ dài `n` **không có hai số `1` đứng liền nhau**. Hãy **cắt tỉa**: không bao giờ đặt `'1'` ngay sau một `'1'`, thay vì sinh hết rồi lọc.

```python
no_consecutive_ones(3) -> ["000", "001", "010", "100", "101"]
```

Sau khi làm xong, hãy đếm kết quả cho `n = 1..10` — bạn có nhận ra dãy số quen thuộc không?

In [31]:
def exercise_6_no_consecutive_ones(n: int) -> List[str]:
    # TODO
    pass


# assert exercise_6_no_consecutive_ones(3) == ["000", "001", "010", "100", "101"]
# assert exercise_6_no_consecutive_ones(1) == ["0", "1"]

# 15. Gợi ý

**Bài 1**
- Mức 1: số kết quả = tích số chữ cái các phím.
- Mức 2: `"79"` = 4 × 4. `"234"` = 3 × 3 × 3.
- Mức 3: backtracking chọn chữ của phím đầu chậm nhất (tầng trên cùng).

**Bài 3**
- Mức 1: `count = 1`, nhân dần `len(PHONE[d])`.
- Mức 2: nhớ `digits` rỗng trả về 0, không phải 1.

**Bài 4**
- Mức 1: so với code mục 7.

**Bài 5**
- Mức 1: giống bài chính, nhưng mọi tầng có cùng lựa chọn `"01"`.
- Mức 2: base case `len(path) == n`.

**Bài 6**
- Mức 1: trước khi `append('1')`, kiểm tra `path` rỗng hoặc `path[-1] == '0'`.
- Mức 2: dãy số kết quả: 2, 3, 5, 8, 13, …

# 16. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [32]:
print(len(sol.letterCombinations("79")), len(sol.letterCombinations("234")))   # 16 27
print(sol.letterCombinations("92")[:5])

16 27
['wa', 'wb', 'wc', 'xa', 'xb']


## Đáp án Bài 2

In [33]:
print(list(product("ab", "xy")))                          # [('a','x'), ('a','y'), ('b','x'), ('b','y')]
print(["".join(t) for t in product("ab", repeat=2)])      # ['aa', 'ab', 'ba', 'bb']
print(len(list(product("abc", "def", "ghi"))))            # 27

[('a', 'x'), ('a', 'y'), ('b', 'x'), ('b', 'y')]
['aa', 'ab', 'ba', 'bb']
27


## Đáp án Bài 3

In [34]:
def answer_3_count_combinations(digits: str) -> int:
    if not digits:
        return 0
    count = 1
    for d in digits:
        count *= len(PHONE[d])
    return count

assert answer_3_count_combinations("23") == 9
assert answer_3_count_combinations("79") == 16
assert answer_3_count_combinations("") == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [35]:
def answer_4_letter_combinations(digits: str) -> List[str]:
    if not digits:
        return []
    result, path = [], []

    def backtrack(index: int):
        if index == len(digits):
            result.append("".join(path))
            return
        for ch in PHONE[digits[index]]:
            path.append(ch)
            backtrack(index + 1)
            path.pop()

    backtrack(0)
    return result

assert sorted(answer_4_letter_combinations("23")) == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert answer_4_letter_combinations("") == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [36]:
def answer_5_binary_strings(n: int) -> List[str]:
    result, path = [], []

    def backtrack():
        if len(path) == n:
            result.append("".join(path))
            return
        for ch in "01":
            path.append(ch)
            backtrack()
            path.pop()

    backtrack()
    return result

assert answer_5_binary_strings(2) == ["00", "01", "10", "11"]
assert answer_5_binary_strings(0) == [""]
assert len(answer_5_binary_strings(5)) == 32
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

In [37]:
def answer_6_no_consecutive_ones(n: int) -> List[str]:
    result, path = [], []

    def backtrack():
        if len(path) == n:
            result.append("".join(path))
            return
        path.append("0")
        backtrack()
        path.pop()
        if not path or path[-1] == "0":      # cắt tỉa: không đặt '1' sau '1'
            path.append("1")
            backtrack()
            path.pop()

    backtrack()
    return result

assert answer_6_no_consecutive_ones(3) == ["000", "001", "010", "100", "101"]
assert answer_6_no_consecutive_ones(1) == ["0", "1"]
print("Bài 6 PASS")
print("Số kết quả n = 1..10:", [len(answer_6_no_consecutive_ones(n)) for n in range(1, 11)])

Bài 6 PASS
Số kết quả n = 1..10: [2, 3, 5, 8, 13, 21, 34, 55, 89, 144]


Dãy `2, 3, 5, 8, 13, 21, …` là **dãy Fibonacci**! Chuỗi độ dài `n` hợp lệ hoặc kết thúc bằng `0` (gắn vào bất kỳ chuỗi hợp lệ độ dài `n-1`), hoặc kết thúc bằng `01` (gắn vào chuỗi hợp lệ độ dài `n-2`).

# 17. Tổng kết

> Khi số vòng `for` lồng nhau phụ thuộc vào input, hãy dùng **Backtracking**: một hàm đệ quy xử lý **một tầng**, với mẫu **chọn → đi tiếp → bỏ chọn**.

---

## Checklist trước khi nộp

- [ ] Xử lý `digits = ""` → `[]`?
- [ ] Bảng phím đúng (7 → `pqrs`, 9 → `wxyz`)?
- [ ] Có base case `index == len(digits)`?
- [ ] Có `path.pop()` sau lời gọi đệ quy?
- [ ] Lưu **bản sao** (`"".join(path)`), không lưu tham chiếu `path`?